In [ ]:
%cd /workspace/moshi-ditto-streaming-debug-2

In [10]:
import os, torch

PROJECT_ROOT = os.path.abspath(os.getcwd())
print(f"📁 Project root: {PROJECT_ROOT}")

# ════════════════════════════════════════════════════
# 🔧 Server settings
# ════════════════════════════════════════════════════
SERVER_HOST = "0.0.0.0"
SERVER_PORT = 7860

# ════════════════════════════════════════════════════
# 🔑 Moshi
# ════════════════════════════════════════════════════
MOSHI_HF_REPO   = "kyutai/moshiko-pytorch-bf16"
MOSHI_WEIGHT    = None   # local path to skip HF download
MIMI_WEIGHT     = None
MOSHI_TOKENIZER = None

# ════════════════════════════════════════════════════
# 🔑 Bridge
# ════════════════════════════════════════════════════
BRIDGE_CKPT   = os.path.join(PROJECT_ROOT, "checkpoints", "bridge_best.pt")   # .pt streaming
BRIDGE_CONFIG = os.path.join(PROJECT_ROOT, "bridge_module", "config.yaml")

# BRIDGE_CHUNK: how many Mimi tokens to batch before running the bridge.
# The adaptive flush timer (BRIDGE_FLUSH_TIMEOUT_MS) ensures partial batches
# are sent promptly even during pauses — so higher chunk sizes no longer
# cause long stalls.
# Recommended: 2 on A100 for lowest latency.
BRIDGE_CHUNK = 2

# Maximum time (ms) to wait before flushing a partial token batch to Ditto.
# 50ms = flush at most 50ms after the first token of a new batch arrives.
BRIDGE_FLUSH_TIMEOUT_MS = 50

# ════════════════════════════════════════════════════
# 🔑 Ditto
# ════════════════════════════════════════════════════
DITTO_DATA_ROOT = os.path.join(PROJECT_ROOT, "ditto-inference",
                                "checkpoints", "ditto_trt_Ampere_Plus")
DITTO_CFG_PKL   = os.path.join(PROJECT_ROOT, "ditto-inference",
                                "checkpoints", "ditto_cfg",
                                "v0.4_hubert_cfg_trt.pkl")
DITTO_EMO             = 4    # emotion index 0-7
DITTO_SAMPLING_STEPS  = 10   # ↓ reduces diffusion latency (try 10–20 for speed)
DITTO_JPEG_QUALITY    = 70   # ↓ smaller JPEG = faster transfer (70 optimal for real-time)

# ════════════════════════════════════════════════════
# ✅ Validate paths
# ════════════════════════════════════════════════════
print("\n📋 Path Check")
print("─" * 60)
errors = []
checks = [
    ("Bridge .pt",       BRIDGE_CKPT),
    ("Bridge config",    BRIDGE_CONFIG),
    ("Ditto data root",  DITTO_DATA_ROOT),
    ("Ditto cfg pkl",    DITTO_CFG_PKL),
]
for label, path in checks:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'}  {label:<22} {path}")
    if not exists:
        errors.append(label)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n  Device          : {DEVICE}")
print(f"  Bridge chunk    : {BRIDGE_CHUNK} tokens ({BRIDGE_CHUNK*80}ms audio)")
print(f"  Bridge flush    : {BRIDGE_FLUSH_TIMEOUT_MS}ms adaptive timeout")
print(f"  Ditto steps     : {DITTO_SAMPLING_STEPS} (lower=faster)")

if errors:
    print(f"\n⚠️  {len(errors)} issue(s): {errors}")
else:
    print("\n✅ All checks passed — ready to launch!")

📁 Project root: /workspace/moshi-ditto-streaming-debug-v1

📋 Path Check
────────────────────────────────────────────────────────────
  ✅  Bridge .pt             /workspace/moshi-ditto-streaming-debug-v1/checkpoints/bridge_best.pt
  ✅  Bridge config          /workspace/moshi-ditto-streaming-debug-v1/bridge_module/config.yaml
  ✅  Ditto data root        /workspace/moshi-ditto-streaming-debug-v1/ditto-inference/checkpoints/ditto_trt_Ampere_Plus
  ✅  Ditto cfg pkl          /workspace/moshi-ditto-streaming-debug-v1/ditto-inference/checkpoints/ditto_cfg/v0.4_hubert_cfg_trt.pkl

  Device          : cuda
  Bridge chunk    : 2 tokens (160ms audio)
  Bridge flush    : 50ms adaptive timeout
  Ditto steps     : 10 (lower=faster)

✅ All checks passed — ready to launch!


---
## 🔬 Cell 4b — Pipeline Import Health Check

Verifies all modules import correctly **before** launching the server.
Run this after any code changes to catch import errors early.

In [11]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.getcwd())
for p in [PROJECT_ROOT,
          os.path.join(PROJECT_ROOT, 'bridge_module'),
          os.path.join(PROJECT_ROOT, 'moshi-inference')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("🔬 Pipeline Import Health Check")
print("═" * 50)

checks = {
    "pipeline.sync_types":        "TaggedToken, TaggedFeatures, TaggedFrame, seq_pack",
    "pipeline.streaming_moshi":   "StreamingMoshiState",
    "pipeline.ditto_stream_adapter": "DittoStreamAdapter",
    "inference":                  "StreamingBridgeInference",
}

all_ok = True
for mod, symbols in checks.items():
    try:
        m = __import__(mod, fromlist=symbols.split(', '))
        for sym in symbols.split(', '):
            getattr(m, sym.strip())
        print(f"  ✅  {mod:<40}  [{symbols}]")
    except Exception as e:
        print(f"  ❌  {mod:<40}  ERROR: {e}")
        all_ok = False

# Check struct.pack for seq encoding
try:
    import struct
    b = struct.pack('>I', 12345)
    assert len(b) == 4
    print(f"  ✅  seq_pack (struct.pack >I)              [4-byte big-endian uint32]")
except Exception as e:
    print(f"  ❌  seq_pack check failed: {e}")
    all_ok = False

print("═" * 50)
print("\n✅ All imports OK — safe to launch server." if all_ok else
      "\n❌ Fix import errors above before launching.")

🔬 Pipeline Import Health Check
══════════════════════════════════════════════════


In file included from /usr/local/lib/python3.11/dist-packages/numpy/_core/include/numpy/ndarraytypes.h:1913,
                 from /usr/local/lib/python3.11/dist-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from /root/.pyxbld/temp.linux-x86_64-cpython-311/workspace/moshi-ditto-streaming-debug-v1/pipeline/../ditto-inference/core/utils/blend/blend.c:1157:
/usr/local/lib/python3.11/dist-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~


  ✅  pipeline.sync_types                       [TaggedToken, TaggedFeatures, TaggedFrame, seq_pack]
  ✅  pipeline.streaming_moshi                  [StreamingMoshiState]
  ✅  pipeline.ditto_stream_adapter             [DittoStreamAdapter]
  ✅  inference                                 [StreamingBridgeInference]
  ✅  seq_pack (struct.pack >I)              [4-byte big-endian uint32]
══════════════════════════════════════════════════

✅ All imports OK — safe to launch server.


---
## 🚀 Cell 5a — Kill Any Existing Server

In [12]:
import os, signal, time

def get_pids_on_port(port):
    hex_port = format(port, '04X')
    inodes = set()
    for path in ["/proc/net/tcp", "/proc/net/tcp6"]:
        try:
            with open(path) as f:
                for line in f.readlines()[1:]:
                    parts = line.split()
                    if len(parts) > 9 and parts[1].split(":")[1] == hex_port:
                        inodes.add(parts[9])
        except FileNotFoundError:
            pass
    pids = []
    for pid in os.listdir("/proc"):
        if not pid.isdigit(): continue
        try:
            for fd in os.listdir(f"/proc/{pid}/fd"):
                try:
                    link = os.readlink(f"/proc/{pid}/fd/{fd}")
                    if "socket:[" in link and link.split("[")[1].rstrip("]") in inodes:
                        pids.append(int(pid)); break
                except OSError: pass
        except OSError: pass
    return pids

print("🔪 Killing any existing server processes...")
killed = []
for pid in os.listdir("/proc"):
    if not pid.isdigit(): continue
    try:
        with open(f"/proc/{pid}/cmdline") as f:
            if "streaming_server.py" in f.read():
                os.kill(int(pid), signal.SIGKILL); killed.append(pid)
    except (OSError, PermissionError): pass

if killed: print(f"   💀 Killed PIDs: {', '.join(killed)}")
time.sleep(1)

remaining = get_pids_on_port(SERVER_PORT)
for pid in remaining:
    try: os.kill(pid, signal.SIGKILL); print(f"   💀 Killed port-holder PID {pid}")
    except: pass
if remaining: time.sleep(1)

if get_pids_on_port(SERVER_PORT):
    print(f"⚠️  Port {SERVER_PORT} still busy!")
else:
    print(f"✅ Port {SERVER_PORT} is free — safe to launch.")

🔪 Killing any existing server processes...
✅ Port 7860 is free — safe to launch.


In [ ]:
!python streaming_server.py --log-level debug

/workspace/moshi-ditto-streaming-debug-v1/streaming_server.py:725: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
INFO:     Started server process [3635]
INFO:     Waiting for application startup.
13:40:45  INFO      streaming_server  ============================================================
13:40:45  INFO      streaming_server    Moshi + Bridge + Ditto — Streaming Server Starting  v3
13:40:45  INFO      streaming_server  ============================================================
13:40:45  INFO      streaming_server    Device : cuda
13:40:45  INFO      streaming_server    cuDNN benchmark + TF32 enabled (A100 optimised)
13:40:45  INFO      streaming_server    PyTorch CPU threads: 8
13:40:45  INFO      streaming_server  
[1/2] Loading Moshi …
13:40:45  INFO      pipeline.streaming_